In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
# Argumentos de estabilidad para Docker
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

try:
    # El Manager instala el driver compatible con la versión de Chrome instalada
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), 
        options=options
    )
    
    driver.get("https://www.google.com")
    print(f"¡CONECTADO! Título de la página: {driver.title}")
    driver.quit()
    
except Exception as e:
    print(f" Error: {e}")

¡CONECTADO! Título de la página: Google


In [5]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN (Configuración de Laboratorio)
# ==========================================================
def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ==========================================================
# 2. CONFIGURACIÓN DEL GRUPO (Cada grupo modifica aquí)
# ==========================================================

# Pegar la URL del sitio asignado (Retail, Energía, E-commerce, etc.)
URL_OBJETIVO = "https://books.toscrape.com/" 

# Definir las etiquetas técnicas (clases CSS) encontradas con el Inspector de Chrome
SELECTOR_CONTENEDOR = ".product_pod" 
SELECTOR_DATO_A     = ".product_pod h3 a"
SELECTOR_DATO_B     = ".price_color"

# ==========================================================
# 3. EJECUCIÓN DEL SCRAPING
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print(f" Conectando a la fuente de datos...")
    driver.get(URL_OBJETIVO)
    time.sleep(5) # Tiempo de espera para carga de datos dinámicos

    # Capturamos todos los bloques de información
    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f" Se encontraron {len(elementos)} registros potenciales.")

    for item in elementos:
        try:
            # Extracción genérica de datos
            columna_a = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
            columna_b = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text
            
            # Limpieza básica de caracteres no numéricos (opcional según el proyecto)
            valor_limpio = "".join(filter(str.isdigit, columna_b))

            dataset_final.append({
                "variable_nombre": columna_a,
                "variable_valor": valor_limpio,
                "status": 1.0 # Indicador de registro procesado (float)
            })
        except:
            # Si un elemento está vacío o es un anuncio, saltar al siguiente
            continue

    # ==========================================================
    # 4. SALIDA DE DATOS (Para análisis posterior)
    # ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_extraidos_grupo.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f" Proceso exitoso. Archivo generado: {nombre_archivo}")
        print(df.head()) # Muestra de los primeros 5 datos
    else:
        print(" No se capturaron datos. Revisar los selectores CSS.")

finally:
    driver.quit()
    print(" Navegador cerrado.")

 Conectando a la fuente de datos...
 Se encontraron 20 registros potenciales.
 Proceso exitoso. Archivo generado: datos_extraidos_grupo.csv
                variable_nombre variable_valor  status
0            A Light in the ...           5177     1.0
1            Tipping the Velvet           5374     1.0
2                    Soumission           5010     1.0
3                 Sharp Objects           4782     1.0
4  Sapiens: A Brief History ...           5423     1.0
 Navegador cerrado.


In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACIÓN ---
datos_totales = []
paginas_objetivo = 5 

try:
    # URL REAL DEL SITIO
    driver.get("https://books.toscrape.com/")

    # --- PASO 3: BUCLE DE PAGINACIÓN ---
    for i in range(paginas_objetivo):
        print(f"--- Procesando Página {i + 1} ---")

        # Espera a que los productos aparezcan
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "product_pod"))
        )
        
        # Captura los elementos de la página actual
        productos = driver.find_elements(By.CLASS_NAME, "product_pod")
        for p in productos:
            datos_totales.append(p.text.replace('\n', ' - ')) 

        # LÓGICA DE NAVEGACIÓN
        try:
            # Buscamos el botón 'next' (en esta web es un enlace <a> con texto 'next')
            boton_siguiente = driver.find_element(By.XPATH, "//a[contains(text(),'next')]")
            boton_siguiente.click()
            time.sleep(2) 
        except Exception:
            print("Se ha llegado a la última página o el botón no es visible.")
            break 

    print(f"\nProceso finalizado con éxito.")
    print(f"Total de registros capturados: {len(datos_totales)}")

except Exception as e:
    print(f"Error detectado durante la ejecución: {e}")

finally:
    # --- PASO 4: CIERRE SEGURO ---
    driver.quit()
    print("Navegador cerrado.")

--- Procesando Página 1 ---
--- Procesando Página 2 ---
--- Procesando Página 3 ---
--- Procesando Página 4 ---
--- Procesando Página 5 ---

Proceso finalizado con éxito.
Total de registros capturados: 100
Navegador cerrado.


In [8]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN (Configuración de Laboratorio)
# ==========================================================
def iniciar_navegador():
    options = Options()
    # Mantenemos tus opciones de Docker/Laboratorio
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ==========================================================
# 2. CONFIGURACIÓN DEL PROYECTO
# ==========================================================
URL_OBJETIVO = "https://books.toscrape.com/" 
paginas_a_recorrer = 3  # Cantidad de páginas que quieres scrapear

# Selectores CSS (Importante: llevan punto para clases)
SELECTOR_CONTENEDOR = ".product_pod" 
SELECTOR_DATO_A     = "h3 a"          # Título
SELECTOR_DATO_B     = ".price_color"  # Precio

# ==========================================================
# 3. EJECUCIÓN DEL SCRAPING (Bucle Integrado)
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print(f"--- Iniciando conexión con {URL_OBJETIVO} ---")
    driver.get(URL_OBJETIVO)

    for i in range(paginas_a_recorrer):
        print(f" Procesando Página {i + 1}...")
        
        # Espera a que los elementos estén presentes
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, SELECTOR_CONTENEDOR))
        )
        
        # Capturamos los bloques de la página actual
        elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)

        for item in elementos:
            try:
                # Extracción de Título y Precio
                nombre = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).get_attribute("title")
                # Si el 'title' está vacío, usamos el texto normal
                if not nombre: nombre = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
                
                precio_raw = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text
                
                # Limpieza: dejamos solo números y puntos para el precio
                precio_limpio = "".join(c for c in precio_raw if c.isdigit() or c == '.')

                dataset_final.append({
                    "Libro": nombre,
                    "Precio": float(precio_limpio),
                    "Pagina_Origen": i + 1
                })
            except Exception as e:
                continue

        # LÓGICA DE NAVEGACIÓN (Saltar a la siguiente página)
        try:
            # Buscamos el enlace que contiene la palabra 'next'
            boton_siguiente = driver.find_element(By.XPATH, "//a[contains(text(),'next')]")
            boton_siguiente.click()
            time.sleep(2) # Pausa para que cargue la nueva página
        except:
            print(" Fin de las páginas disponibles.")
            break

    # ==========================================================
    # 4. SALIDA DE DATOS
    # ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_extraidos_libros.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f"\nProceso exitoso. Archivo generado: {nombre_archivo}")
        print(f"Total de libros capturados: {len(df)}")
        print("\n--- Muestra de los primeros datos ---")
        print(df.head())
    else:
        print("No se capturaron datos. Revisa los selectores.")

except Exception as e:
    print(f"Error crítico: {e}")

finally:
    driver.quit()
    print("Navegador cerrado.")

--- Iniciando conexión con https://books.toscrape.com/ ---
 Procesando Página 1...
 Procesando Página 2...
 Procesando Página 3...

Proceso exitoso. Archivo generado: datos_extraidos_libros.csv
Total de libros capturados: 60

--- Muestra de los primeros datos ---
                                   Libro  Precio  Pagina_Origen
0                   A Light in the Attic   51.77              1
1                     Tipping the Velvet   53.74              1
2                             Soumission   50.10              1
3                          Sharp Objects   47.82              1
4  Sapiens: A Brief History of Humankind   54.23              1
Navegador cerrado.
